# Phase 4 — CJPE: sampled overlapping chunks + mixed-precision LoRMA


In [1]:
%pip install -q huggingface_hub tokenizers scikit-learn tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'encoder_decoder_mixed'
MODEL_PATH = 'checkpoints/encoder_decoder_mixed/model.pt'
TOKENIZER_PATH = 'tokenizer_encoder_decoder/tokenizer.json'
DIM = 640
ENCODER_LAYERS = 6
DECODER_LAYERS = 6
N_HEADS = 16
N_KV_HEADS = 4
FF_DIM = 1728
MAX_ENCODER_LEN = 512
MAX_DECODER_LEN = 256
RMS_NORM_EPS = 1e-6
ROPE_BASE = 10000
GRAD_CLIP = 5.0

MODEL_CONFIG = dict(
    dim=DIM,
    encoder_layers=ENCODER_LAYERS,
    decoder_layers=DECODER_LAYERS,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    ff_dim=FF_DIM,
    max_encoder_len=MAX_ENCODER_LEN,
    max_decoder_len=MAX_DECODER_LEN,
)
MAX_SEQ_LEN = 512
CHUNK_OVERLAP = 16  # number of shared tokens, NOT the stride
CHUNK_ENCODE_BATCH_SIZE = 4  # complete cases per DataLoader batch  # chunks per backbone call
USE_GRADIENT_CHECKPOINTING = False  # recompute each chunk microbatch during backward
UNFREEZE_LAST_N_BLOCKS = 0  # frozen backbone; increase only for partial full fine-tuning
TRAIN_MAX_CHUNKS = 2
EVAL_MAX_CHUNKS = 4
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_4_CJPE'
LABEL_KEY = 'label'  # set to the actual binary label field if the released schema differs

BATCH_SIZE = 4  # complete cases per DataLoader batch
GRAD_ACCUM_STEPS = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 5
PATIENCE = 10
MIN_DELTA = 0.001
ADAPTER_CONFIG = dict(r=16, alpha=32, dropout=0.05, do_pi=True)
HEAD_DROPOUT = 0.1
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'lorma'

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = None  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = True  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/rr_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'cjpe_runs'
SAVE_TAG = f'cjpe_{PHASE1_TAG}_{TECHNIQUE}_len{MAX_SEQ_LEN}_ov{CHUNK_OVERLAP}_sample{TRAIN_MAX_CHUNKS}_eval{EVAL_MAX_CHUNKS}'
if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'cjpe_{PHASE1_TAG}_{TECHNIQUE}_len{MAX_SEQ_LEN}_ov{CHUNK_OVERLAP}_sample{TRAIN_MAX_CHUNKS}_eval{EVAL_MAX_CHUNKS}'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = dict(save_tag=SAVE_TAG, best_tag=BEST_TAG, technique=TECHNIQUE,
                    phase1_tag=PHASE1_TAG, model_path=MODEL_PATH,
                    tokenizer_path=TOKENIZER_PATH, submission_zip=SUBMISSION_ZIP_NAME,
                    train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                    use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                    max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP, input_format='overlapping_chunks_v1',
                    pooling='token_mean_then_chunk_mean',
                    unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

print('Chunk length/overlap:', MAX_SEQ_LEN, CHUNK_OVERLAP)


Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/encoder_decoder_mixed/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_encoder_decoder/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/cjpe_encoder_decoder_mixed_lorma_len512_ov16_sample2_eval4/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/cjpe_encoder_decoder_mixed_lorma_len512_ov16_sample2_eval4_best/checkpoint.pt
Submission ZIP: /kaggle/working/rr_runs/cjpe_encoder_decoder_mixed_lorma_len512_ov16_sample2_eval4/cjpe_encoder_decoder_mixed_lorma_len512_ov16_sample2_eval4.zip
Chunk length/overlap: 512 16


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID
LABELS = [0, 1]

def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def normalize_documents(raw, labelled):
    if isinstance(raw, list): docs = raw
    elif isinstance(raw, dict) and 'id' in raw and 'text' in raw: docs = [raw]
    elif isinstance(raw, dict):
        docs = []
        for doc_id, value in raw.items():
            if not isinstance(value, dict): raise ValueError('Expected ID -> case object mapping')
            doc = dict(value)
            doc.setdefault('id', doc_id)
            if doc['id'] != doc_id: raise ValueError('Document ID mismatch')
            docs.append(doc)
    else: raise ValueError('Expected a list of cases or ID -> case mapping')
    result = []
    for row_index, doc in enumerate(docs):
        case_id, text = doc['id'], doc['text']
        if not isinstance(text, str): raise ValueError(f'{case_id}: text must be a string')
        row = dict(id=case_id, text=text, row_index=row_index)
        if labelled:
            if LABEL_KEY not in doc:
                raise ValueError(f'Missing label field {LABEL_KEY!r}; available fields: {list(doc)}. Set LABEL_KEY.')
            label = doc[LABEL_KEY]
            if type(label) is not int or label not in (0, 1):
                raise ValueError(f'{case_id}: expected integer label 0/1, got {label!r}')
            row['label'] = label
        result.append(row)
    if not result: raise ValueError('Empty split')
    return result

# Keep each input record separate, even when IDs repeat.
# Test is loaded later, only for prediction.
data = {}
for split in ['train', 'dev']:
    data[split] = normalize_documents(
        download_json(f'{DATA_PREFIX}/Train/{split}.json'), labelled=True)
    counts = Counter(d['id'] for d in data[split])
    print(split, '| records:', len(data[split]), '| unique IDs:', len(counts),
          '| repeated IDs:', sum(n > 1 for n in counts.values()))
    print('Class counts:', dict(Counter(d['label'] for d in data[split])))


(…)tokenizer_encoder_decoder/tokenizer.json:   0%|          | 0.00/2.39M [00:00<?, ?B/s]

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

train | records: 37387 | unique IDs: 32305 | repeated IDs: 5082
Class counts: {0: 22067, 1: 15320}


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 80.3MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

dev | records: 3505 | unique IDs: 2511 | repeated IDs: 994
Class counts: {0: 1749, 1: 1756}


In [4]:
from sklearn.model_selection import train_test_split

case_labels = {doc['id']: doc['label'] for doc in data['train']}
case_ids = list(case_labels)

selected_ids, _ = train_test_split(
    case_ids,
    train_size=0.20,
    random_state=SEED,
    stratify=[case_labels[case_id] for case_id in case_ids],
)
selected_ids = set(selected_ids)

original_size = len(data['train'])
data['train'] = [
    doc for doc in data['train']
    if doc['id'] in selected_ids
]

print(f"Training records: {original_size} → {len(data['train'])}")
print("Class counts:", dict(Counter(d['label'] for d in data['train'])))

Training records: 37387 → 7463
Class counts: {0: 4393, 1: 3070}


In [5]:
def chunk_token_ids(tokens, max_len=MAX_SEQ_LEN, overlap=CHUNK_OVERLAP):
    if not 0 <= overlap < max_len: raise ValueError('Invalid chunk overlap')
    if not tokens: return [[EMPTY_ID]]
    chunks = []
    for start in range(0, len(tokens), max_len - overlap):
        chunks.append(tokens[start:start + max_len])
        if start + max_len >= len(tokens): break
    return chunks

def select_chunk_indices(count, limit, training, seed):
    if limit is None or count <= limit:
        return list(range(count))
    if limit < 1: raise ValueError('Chunk limit must be positive or None')
    if training:
        rng = random.Random(seed)
        # One random chunk from each equal-sized region; two regions = document halves.
        return [rng.randrange(i * count // limit, (i + 1) * count // limit)
                for i in range(limit)]
    if limit == 1: return [count // 2]
    return [i * (count - 1) // (limit - 1) for i in range(limit)]

class CJPECaseDataset(Dataset):
    def __init__(self, documents, split):
        self.split, self.epoch = split, 0
        self.rows = []
        token_lengths, chunk_counts = [], []
        for doc in tqdm(documents, desc='Tokenizing cases'):
            tokens = tokenizer.encode(doc['text'], add_special_tokens=False).ids
            chunks = chunk_token_ids(tokens)
            self.rows.append(dict(doc_id=doc['id'], chunks=chunks, label=doc.get('label', -100)))
            token_lengths.append(len(tokens)); chunk_counts.append(len(chunks))
        lengths = torch.tensor(token_lengths, dtype=torch.float)
        print('Cases/chunks:', len(self.rows), sum(chunk_counts), 'max chunks/case:', max(chunk_counts))
        print('Token lengths p50/p90/p99/max:', torch.quantile(lengths, torch.tensor([.5,.9,.99,1.])).tolist())
    def __len__(self): return len(self.rows)
    def set_epoch(self, epoch): self.epoch = epoch
    def __getitem__(self, index):
        row = self.rows[index]
        training = self.split == 'train'
        limit = TRAIN_MAX_CHUNKS if training else EVAL_MAX_CHUNKS
        selected = select_chunk_indices(len(row['chunks']), limit, training,
                                        SEED + self.epoch * len(self.rows) + index)
        return dict(row, chunks=[row['chunks'][i] for i in selected])

def collate_cjpe(rows):
    chunks, owners = [], []
    for case_index, row in enumerate(rows):
        chunks.extend(row['chunks'])
        owners.extend([case_index] * len(row['chunks']))
    lengths = torch.tensor([len(c) for c in chunks], dtype=torch.long)
    ids = torch.full((len(chunks), int(lengths.max())), PAD_ID, dtype=torch.long)
    for i, chunk in enumerate(chunks): ids[i, :len(chunk)] = torch.tensor(chunk)
    return dict(input_ids=ids, lengths=lengths, chunk_to_case=torch.tensor(owners, dtype=torch.long),
                labels=torch.tensor([r['label'] for r in rows], dtype=torch.long),
                doc_ids=[r['doc_id'] for r in rows])

datasets = {split: CJPECaseDataset(docs, split) for split, docs in data.items()}
def make_loader(split):
    return DataLoader(datasets[split], batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                      collate_fn=collate_cjpe, num_workers=0, pin_memory=(DEVICE == 'cuda'))


Tokenizing cases:   0%|          | 0/7463 [00:00<?, ?it/s]

Cases/chunks: 7463 64913 max chunks/case: 947
Token lengths p50/p90/p99/max: [2676.0, 8285.0, 23645.9375, 469595.0]


Tokenizing cases:   0%|          | 0/3505 [00:00<?, ?it/s]

Cases/chunks: 3505 37822 max chunks/case: 124
Token lengths p50/p90/p99/max: [3890.0, 9496.599609375, 25317.25390625, 61026.0]


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F



# ============================================================
# RMSNorm
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=RMS_NORM_EPS):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = (x*x).mean(
            dim=-1,
            keepdim=True
        )

        return self.weight * x * torch.rsqrt(
            rms + self.eps
        )


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=ROPE_BASE,
    ):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2,
                    dtype=torch.float32,
                ) / head_dim
            )
        )

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32,
        )

        freqs = torch.outer(
            positions,
            inv_freq,
        )

        # [T, head_dim]
        emb = torch.cat(
            [freqs, freqs],
            dim=-1,
        )

        self.register_buffer(
            "cos",
            emb.cos()[None, None, :, :],
            persistent=False,
        )

        self.register_buffer(
            "sin",
            emb.sin()[None, None, :, :],
            persistent=False,
        )

    def forward(self, seq_len):

        return (
            self.cos[:, :, :seq_len],
            self.sin[:, :, :seq_len],
        )


def rotate_half(x):

    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]

    return torch.cat(
        [-x2, x1],
        dim=-1,
    )


def apply_rope(q, k, cos, sin):

    q = (
        q * cos
        +
        rotate_half(q) * sin
    )

    k = (
        k * cos
        +
        rotate_half(k) * sin
    )

    return q, k


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):

    def __init__(
        self,
        dim,
        hidden_dim,
    ):
        super().__init__()

        self.gate_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.up_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.down_proj = nn.Linear(
            hidden_dim,
            dim,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(
                self.gate_proj(x)
            )
            *
            self.up_proj(x)
        )


# ============================================================
# GQA + Flash Attention
# ============================================================

class Attention(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        max_seq_len,
        causal=False,
        use_rope=True,
    ):
        super().__init__()

        assert dim % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.dim = dim
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads

        self.head_dim = dim // n_heads

        self.causal = causal
        self.use_rope = use_rope

        # ----------------------------------------------------
        # Q / K / V projections
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            dim,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.o_proj = nn.Linear(
            n_heads * self.head_dim,
            dim,
            bias=False,
        )

        # RoPE only needs to be constructed for self-attention
        if use_rope:

            self.rope = RotaryEmbedding(
                self.head_dim,
                max_seq_len,
            )

    def forward(
        self,
        x,
        kv=None,
        past_kv=None,
        use_cache=False,
    ):

        B, T, C = x.shape

        # ====================================================
        # Queries
        # ====================================================

        q = self.q_proj(x)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        )

        q = q.transpose(1, 2)

        # ====================================================
        # Keys / Values
        # ====================================================

        # Self-attention:
        #     source = x
        #
        # Cross-attention:
        #     source = encoder output

        source = x if kv is None else kv

        S = source.shape[1]

        k = self.k_proj(source)
        v = self.v_proj(source)

        k = k.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        v = v.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # ====================================================
        # RoPE
        # ====================================================

        # Only self-attention gets RoPE.
        # Cross-attention does NOT use RoPE.

        if kv is None and self.use_rope:

            current_len = k.shape[2]

            cos, sin = self.rope(
                current_len
            )

            q, k = apply_rope(
                q,
                k,
                cos,
                sin,
            )

        # ====================================================
        # Decoder KV Cache
        # ====================================================

        if (
            kv is None
            and past_kv is not None
        ):

            past_k, past_v = past_kv

            k = torch.cat(
                [past_k, k],
                dim=2,
            )

            v = torch.cat(
                [past_v, v],
                dim=2,
            )

        # ====================================================
        # GQA
        # ====================================================

        if self.n_kv_heads != self.n_heads:

            repeat_factor = (
                self.n_heads
                // self.n_kv_heads
            )

            k = k.repeat_interleave(
                repeat_factor,
                dim=1,
            )

            v = v.repeat_interleave(
                repeat_factor,
                dim=1,
            )

        # ====================================================
        # Flash / SDPA Attention
        # ====================================================

        # During cached decoding, only one new token is
        # queried, so causal masking is unnecessary.

        is_causal = (
            self.causal
            and kv is None
            and past_kv is None
        )

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=0.0,
            is_causal=is_causal,
        )

        # ====================================================
        # Output projection
        # ====================================================

        out = out.transpose(
            1,
            2,
        ).contiguous()

        out = out.view(
            B,
            T,
            C,
        )

        out = self.o_proj(out)

        # Cache only decoder self-attention
        new_kv = None

        if use_cache and kv is None:
            new_kv = (k, v)

        return out, new_kv


# ============================================================
# Encoder Block
# ============================================================

class EncoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_seq_len,
    ):
        super().__init__()

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_seq_len,
            causal=False,
            use_rope=True,
        )

        self.norm2 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(self, x):

        # Pre-norm self-attention
        x = x + self.self_attn(
            self.norm1(x)
        )[0]

        # Pre-norm FFN
        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_self_attn_len,
        max_cross_attn_len,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Causal self-attention
        # ----------------------------------------------------

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_self_attn_len,
            causal=True,
            use_rope=True,
        )

        # ----------------------------------------------------
        # Encoder-decoder cross-attention
        # ----------------------------------------------------

        self.norm2 = RMSNorm(dim)

        self.cross_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_cross_attn_len,
            causal=False,
            use_rope=False,
        )

        # ----------------------------------------------------
        # FFN
        # ----------------------------------------------------

        self.norm3 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(
        self,
        x,
        encoder_output,
        past_kv=None,
        use_cache=False,
    ):

        # ====================================================
        # 1. Decoder causal self-attention
        # ====================================================

        residual = x

        x_norm = self.norm1(x)

        self_out, new_kv = self.self_attn(
            x_norm,
            past_kv=past_kv,
            use_cache=use_cache,
        )

        x = residual + self_out

        # ====================================================
        # 2. Cross-attention
        # ====================================================

        residual = x

        x_norm = self.norm2(x)

        cross_out, _ = self.cross_attn(
            x_norm,
            kv=encoder_output,
        )

        x = residual + cross_out

        # ====================================================
        # 3. FFN
        # ====================================================

        x = x + self.ffn(
            self.norm3(x)
        )

        return x, new_kv


# ============================================================
# Encoder-Decoder Language Model
# ============================================================

class EncoderDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        dim=DIM,
        encoder_layers=ENCODER_LAYERS,
        decoder_layers=DECODER_LAYERS,
        n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS,
        ff_dim=FF_DIM,
        max_encoder_len=MAX_ENCODER_LEN,
        max_decoder_len=MAX_DECODER_LEN,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # ====================================================
        # Shared token embedding
        # ====================================================

        self.token_embedding = nn.Embedding(
            vocab_size,
            dim,
        )

        # ====================================================
        # Encoder
        # ====================================================

        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_seq_len=max_encoder_len,
            )
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = RMSNorm(dim)

        # ====================================================
        # Decoder
        # ====================================================

        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_self_attn_len=256,
                max_cross_attn_len=max_encoder_len,
            )
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = RMSNorm(dim)

        # ====================================================
        # Output head
        # ====================================================

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False,
        )

        # Tie output projection to token embedding
        self.lm_head.weight = (
            self.token_embedding.weight
        )

    # ========================================================
    # ENCODER
    # ========================================================

    def encode(self, encoder_ids):

        x = self.token_embedding(
            encoder_ids
        )

        for block in self.encoder_blocks:
            x = block(x)

        x = self.encoder_norm(x)

        return x

    # ========================================================
    # DECODER
    # ========================================================

    def decode(
        self,
        decoder_ids,
        encoder_output,
        past_kv=None,
        use_cache=False,
    ):

        x = self.token_embedding(
            decoder_ids
        )

        new_cache = []

        for i, block in enumerate(
            self.decoder_blocks
        ):

            layer_past = None

            if past_kv is not None:
                layer_past = past_kv[i]

            x, layer_cache = block(
                x=x,
                encoder_output=encoder_output,
                past_kv=layer_past,
                use_cache=use_cache,
            )

            if use_cache:
                new_cache.append(
                    layer_cache
                )

        x = self.decoder_norm(x)

        logits = self.lm_head(x)

        if use_cache:
            return logits, new_cache

        return logits, None

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        encoder_ids,
        decoder_ids,
        labels=None,
    ):

        # Encode 512-token context
        encoder_output = self.encode(
            encoder_ids
        )

        # Decode continuation
        logits, _ = self.decode(
            decoder_ids,
            encoder_output,
        )

        # No labels -> inference
        if labels is None:
            return logits

        # ====================================================
        # Language-model loss
        # ====================================================

        loss = F.cross_entropy(
            logits.reshape(
                -1,
                self.vocab_size,
            ),
            labels.reshape(-1),
            ignore_index=-100,
        )

        return logits, loss

    # ========================================================
    # GENERATION
    # ========================================================

    @torch.no_grad()
    def generate(
        self,
        encoder_ids,
        bos_id,
        eos_id=None,
        max_new_tokens=128,
        temperature=1.0,
        top_p=0.9,
    ):

        self.eval()

        # ----------------------------------------------------
        # Encode context ONCE
        # ----------------------------------------------------

        encoder_output = self.encode(
            encoder_ids
        )

        # ----------------------------------------------------
        # Start decoder
        # ----------------------------------------------------

        B = encoder_ids.size(0)

        decoder_ids = torch.full(
            (B, 1),
            bos_id,
            dtype=torch.long,
            device=encoder_ids.device,
        )

        past_kv = None

        # ----------------------------------------------------
        # Autoregressive generation
        # ----------------------------------------------------

        for step in range(max_new_tokens):

            # Only feed the newest token once cache exists
            if past_kv is None:
                current_ids = decoder_ids
            else:
                current_ids = decoder_ids[:, -1:]

            logits, past_kv = self.decode(
                decoder_ids=current_ids,
                encoder_output=encoder_output,
                past_kv=past_kv,
                use_cache=True,
            )

            next_logits = logits[:, -1, :]

            # Temperature
            if temperature != 1.0:
                next_logits = (
                    next_logits / temperature
                )

            # ------------------------------------------------
            # Top-p sampling
            # ------------------------------------------------

            probs = F.softmax(
                next_logits,
                dim=-1,
            )

            sorted_probs, sorted_indices = torch.sort(
                probs,
                descending=True,
                dim=-1,
            )

            cumulative_probs = torch.cumsum(
                sorted_probs,
                dim=-1,
            )

            remove = (
                cumulative_probs > top_p
            )

            # Keep at least the first token
            remove[:, 0] = False

            sorted_probs[remove] = 0.0

            sorted_probs = (
                sorted_probs
                /
                sorted_probs.sum(
                    dim=-1,
                    keepdim=True,
                )
            )

            next_token = torch.multinomial(
                sorted_probs,
                num_samples=1,
            )

            next_token = torch.gather(
                sorted_indices,
                -1,
                next_token,
            )

            decoder_ids = torch.cat(
                [
                    decoder_ids,
                    next_token,
                ],
                dim=1,
            )

            if (
                eos_id is not None
                and torch.all(
                    next_token == eos_id
                )
            ):
                break

        # Remove BOS
        return decoder_ids[:, 1:]


# ============================================================
# Parameter count
# ============================================================



def build_model(vocab_size):
    return EncoderDecoderLM(vocab_size=vocab_size, **MODEL_CONFIG)


In [7]:
class LoRMALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=32, dropout=0.05, do_pi=True):
        super().__init__()
        self.original_linear = original_linear
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.scaling, self.do_pi = alpha / r, do_pi
        self.dropout = nn.Dropout(dropout)
        for p in original_linear.parameters(): p.requires_grad = False
        W = original_linear.weight
        self.A = nn.Parameter(W.new_empty(r, self.out_features))
        self.B = nn.Parameter(W.new_zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def effective_weight(self):
        W = self.original_linear.weight
        delta = self.B @ self.A
        if self.do_pi:
            rows = torch.arange(self.out_features, device=W.device)[:, None]
            cols = torch.arange(self.out_features, device=W.device)[None, :]
            delta = delta.gather(1, (cols - rows) % self.out_features)
        return W + self.scaling * (delta @ W)
    def forward(self, x):
        W = self.original_linear.weight
        W_eff = self.effective_weight()
        return F.linear(x, W, self.original_linear.bias) + F.linear(self.dropout(x), W_eff - W)

from torch.utils.checkpoint import checkpoint as activation_checkpoint

from torch.utils.checkpoint import checkpoint as activation_checkpoint


LORMA_LAST_N_BLOCKS = 2


# Preserve mixed precision in the attached model's RoPE implementation.
def apply_rope(q, k, cos, sin):
    q_cos, q_sin = cos.to(q.dtype), sin.to(q.dtype)
    k_cos, k_sin = cos.to(k.dtype), sin.to(k.dtype)

    return (
        q * q_cos + rotate_half(q) * q_sin,
        k * k_cos + rotate_half(k) * k_sin,
    )


class EfficientDecoderLMForCJPE(EncoderDecoderLM):
    def __init__(self, vocab_size, head_dropout=0.1, **kwargs):
        super().__init__(vocab_size=vocab_size, **kwargs)

        del self.lm_head

        self.d_model = self.dim
        self.cjpe_dropout = nn.Dropout(head_dropout)
        self.cjpe_head = nn.Linear(self.dim, 2)
        self.frozen_decoder_prefix = len(self.decoder_blocks)

        # Extend nonpersistent encoder RoPE caches if the chunk window
        # exceeds the configured encoder length. No weights are changed.
        for block in self.encoder_blocks:
            attention = block.self_attn
            cached_length = attention.rope.cos.size(2)

            if MAX_SEQ_LEN > cached_length:
                attention.rope = RotaryEmbedding(
                    attention.head_dim,
                    max_seq_len=MAX_SEQ_LEN,
                )

    def encode_equal_length_chunks(self, input_ids):
        # Encoder weights and token embeddings remain frozen.
        with torch.no_grad():
            memory = self.encode(input_ids)

            # One pretrained EOS embedding acts as a decoder query.
            query_ids = torch.full(
                (input_ids.size(0), 1),
                EMPTY_ID,
                dtype=torch.long,
                device=input_ids.device,
            )
            x = self.token_embedding(query_ids)

            for index in range(self.frozen_decoder_prefix):
                x, _ = self.decoder_blocks[index](
                    x,
                    encoder_output=memory,
                    use_cache=False,
                )

        for index in range(
            self.frozen_decoder_prefix,
            len(self.decoder_blocks),
        ):
            x, _ = self.decoder_blocks[index](
                x,
                encoder_output=memory,
                use_cache=False,
            )

        x = self.decoder_norm(x)
        return x[:, 0].float()

    def encode_chunks(self, input_ids, lengths):
        # The attached encoder has bidirectional attention but no padding
        # mask. Group by exact length and remove padding before encoding,
        # so padding cannot influence real tokens or cross-attention.
        pooled = self.cjpe_head.weight.new_zeros(
            input_ids.size(0),
            self.dim,
            dtype=torch.float32,
        )

        for length in lengths.unique(sorted=True).tolist():
            indices = torch.nonzero(
                lengths == length,
                as_tuple=True,
            )[0]

            ids = input_ids.index_select(0, indices)[:, :length]
            vectors = self.encode_equal_length_chunks(ids)

            pooled = pooled.index_copy(0, indices, vectors)

        return pooled

    def forward(
        self,
        input_ids,
        lengths,
        chunk_to_case,
        num_cases,
    ):
        device = self.cjpe_head.weight.device
        pooled_chunks = []

        for start in range(
            0,
            input_ids.size(0),
            CHUNK_ENCODE_BATCH_SIZE,
        ):
            end = min(
                start + CHUNK_ENCODE_BATCH_SIZE,
                input_ids.size(0),
            )

            lens = lengths[start:end].to(
                device,
                non_blocking=True,
            )
            width = int(lengths[start:end].max())

            ids = input_ids[start:end, :width].to(
                device,
                non_blocking=True,
            )

            if self.training and USE_GRADIENT_CHECKPOINTING:
                vectors = activation_checkpoint(
                    self.encode_chunks,
                    ids,
                    lens,
                    use_reentrant=False,
                    preserve_rng_state=True,
                )
            else:
                vectors = self.encode_chunks(ids, lens)

            pooled_chunks.append(vectors)

        chunk_vectors = torch.cat(pooled_chunks, dim=0)

        owners = chunk_to_case.to(
            device,
            non_blocking=True,
        )

        sums = chunk_vectors.new_zeros(
            num_cases,
            self.dim,
        ).index_add(
            0,
            owners,
            chunk_vectors,
        )

        counts = torch.bincount(
            owners,
            minlength=num_cases,
        ).to(chunk_vectors.dtype).unsqueeze(1)

        document_vectors = sums / counts.clamp_min(1)

        return self.cjpe_head(
            self.cjpe_dropout(document_vectors)
        )


def forward_batch(model, batch):
    with autocast_context():
        return model(
            batch['input_ids'],
            batch['lengths'],
            batch['chunk_to_case'],
            len(batch['doc_ids']),
        )


phase1_path = download_bucket_file(MODEL_PATH)


def build_cjpe_model():
    # MODEL_CONFIG must be the encoder-decoder configuration
    # matching the selected Phase-1 checkpoint.
    model = EfficientDecoderLMForCJPE(
        vocab_size=VOCAB_SIZE,
        head_dropout=HEAD_DROPOUT,
        **MODEL_CONFIG,
    )

    checkpoint = torch.load(
        phase1_path,
        map_location='cpu',
        weights_only=False,
    )

    if 'model_state_dict' in checkpoint:
        state = checkpoint['model_state_dict']
    elif 'model' in checkpoint:
        state = checkpoint['model']
    elif 'state_dict' in checkpoint:
        state = checkpoint['state_dict']
    elif all(
        isinstance(value, torch.Tensor)
        for value in checkpoint.values()
    ):
        state = checkpoint
    else:
        raise ValueError(
            f'Cannot identify model weights. '
            f'Checkpoint keys: {list(checkpoint)}'
        )

    state = dict(state)
    state.pop('lm_head.weight', None)

    missing, unexpected = model.load_state_dict(
        state,
        strict=False,
    )

    assert set(missing) == {
        'cjpe_head.weight',
        'cjpe_head.bias',
    } and not unexpected, (missing, unexpected)

    del state, checkpoint

    for parameter in model.parameters():
        parameter.requires_grad = False

    adapter_count = min(
        LORMA_LAST_N_BLOCKS,
        len(model.decoder_blocks),
    )

    if adapter_count:
        for block in model.decoder_blocks[-adapter_count:]:
            for attention in [
                block.self_attn,
                block.cross_attn,
            ]:
                for name in [
                    'q_proj',
                    'k_proj',
                    'v_proj',
                    'o_proj',
                ]:
                    setattr(
                        attention,
                        name,
                        LoRMALinear(
                            getattr(attention, name),
                            **ADAPTER_CONFIG,
                        ),
                    )

    # Optional full unfreezing applies to decoder blocks.
    # Leave at 0 for LoRMA-only adaptation.
    if UNFREEZE_LAST_N_BLOCKS:
        for block in model.decoder_blocks[
            -UNFREEZE_LAST_N_BLOCKS:
        ]:
            for parameter in block.parameters():
                parameter.requires_grad = True

    for parameter in model.cjpe_head.parameters():
        parameter.requires_grad = True

    model.frozen_decoder_prefix = len(model.decoder_blocks)

    for index, block in enumerate(model.decoder_blocks):
        if any(
            parameter.requires_grad
            for parameter in block.parameters()
        ):
            model.frozen_decoder_prefix = index
            break

    model = model.to(DEVICE)

    print('Encoder: frozen')
    print(
        'Frozen decoder prefix:',
        model.frozen_decoder_prefix,
    )
    print('LoRMA decoder blocks:', adapter_count)
    print(
        'Total/trainable parameters:',
        sum(p.numel() for p in model.parameters()),
        sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        ),
    )

    return model

(…)eckpoints/encoder_decoder_mixed/model.pt:   0%|          | 0.00/318M [00:00<?, ?B/s]

In [8]:
def experiment_metadata(domain='CJPE'):
    return dict(domain=domain, phase1_tag=PHASE1_TAG, model_path=MODEL_PATH, tokenizer_path=TOKENIZER_PATH,
                labels=LABELS, model_config=MODEL_CONFIG, adapter_config=ADAPTER_CONFIG,
                max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP,
                chunk_encode_batch_size=CHUNK_ENCODE_BATCH_SIZE,
                gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
                train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                sampling='stratified_epoch_seed_v1', use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS,
                pooling='token_mean_then_chunk_mean', input_format='overlapping_chunks_v1',
                data_repo=DATA_REPO, data_prefix=DATA_PREFIX, label_key=LABEL_KEY,
                technique='LoRMA_identity_init', head_dropout=HEAD_DROPOUT,
                seed=SEED, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                patience=PATIENCE, min_delta=MIN_DELTA)

def save_cjpe_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_cjpe_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists() or restore_rng:
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    assert payload['metadata'] == experiment_metadata(domain), 'Checkpoint configuration mismatch'
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload

@torch.no_grad()
def evaluate_cjpe(model, loader):
    model.eval()
    targets, predictions = [], []
    total_loss = total_n = 0
    for batch in loader:
        y = batch['labels'].to(DEVICE)
        logits = forward_batch(model, batch)
        total_loss += F.cross_entropy(logits.float(), y, reduction='sum').item()
        total_n += len(y)
        targets.extend(y.cpu().tolist())
        predictions.extend(logits.argmax(-1).cpu().tolist())
    assert total_n > 0, 'Development set has no cases'
    p, r, f, support = precision_recall_fscore_support(targets, predictions, labels=[0, 1], zero_division=0)
    return dict(loss=total_loss / total_n, macro_f1=float(f.mean()), macro_precision=float(p.mean()),
                macro_recall=float(r.mean()), per_label={label: dict(precision=float(p[i]), recall=float(r[i]),
                f1=float(f[i]), support=int(support[i])) for i, label in enumerate(LABELS)},
                confusion_matrix=confusion_matrix(targets, predictions, labels=[0, 1]).tolist())


In [9]:
def train_cjpe(domain='CJPE'):
    assert domain == 'CJPE'
    assert_run_destination()
    seed_everything(SEED)
    model = build_cjpe_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    tag = SAVE_TAG
    best_tag = BEST_TAG
    start = 0
    extra = dict(best_f1=-1.0, patience_reference=-1.0, stale=0, history=[], best_tag=best_tag)
    if RESUME:
        checkpoint = load_cjpe_checkpoint(SAVE_TAG, model, domain, optimizer, restore_rng=True)
        start, extra = checkpoint['epoch'], checkpoint['extra']
        best_tag = extra['best_tag']
        if 'scaler' in extra: scaler.load_state_dict(extra['scaler'])
        del checkpoint
    train_loader, dev_loader = make_loader('train'), make_loader('dev')
    assert len(train_loader) > 0
    for epoch in range(start, NUM_EPOCHS):
        if extra['stale'] >= PATIENCE: break
        datasets['train'].set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_sum = train_n = group_n = 0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{domain} epoch {epoch+1}')):
            y = batch['labels'].to(DEVICE)
            logits = forward_batch(model, batch)
            loss_sum = F.cross_entropy(logits.float(), y, reduction='sum')
            scaler.scale(loss_sum).backward()
            group_n += len(y)
            train_n += len(y)
            train_sum += loss_sum.item()
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None: p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                group_n = 0
        metrics = evaluate_cjpe(model, dev_loader)
        f1 = metrics['macro_f1']
        improved = f1 > extra['best_f1']
        if improved: extra['best_f1'] = f1
        if f1 > extra['patience_reference'] + MIN_DELTA:
            extra['patience_reference'], extra['stale'] = f1, 0
        else: extra['stale'] += 1
        extra['history'].append(dict(epoch=epoch+1, train_loss=train_sum/train_n, **metrics))
        print(f"CJPE train loss={train_sum/train_n:.4f}, dev macro-F1={f1:.4f}, precision={metrics['macro_precision']:.4f}, recall={metrics['macro_recall']:.4f}, patience={extra['stale']}/{PATIENCE}")
        extra['scaler'] = scaler.state_dict()
        if improved: save_cjpe_checkpoint(best_tag, model, optimizer, epoch+1, extra, domain)
        save_cjpe_checkpoint(tag, model, optimizer, epoch+1, extra, domain)
    load_cjpe_checkpoint(best_tag, model, domain)
    metrics = evaluate_cjpe(model, dev_loader)
    with open(WORK_DIR / f'{domain}_dev_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
    with open(WORK_DIR / f'{domain}_history.json', 'w') as f: json.dump(extra['history'], f, indent=2)
    print('Best development metrics:', metrics)
    del optimizer
    return model, extra['history']


In [10]:
def validate_predictions(predictions, documents):
    assert len(predictions) == len(documents)
    assert [p['id'] for p in predictions] == [d['id'] for d in documents]
    assert len({p['id'] for p in predictions}) == len(predictions)
    assert all(set(p) == {'id', 'pred'} and type(p['pred']) is int and p['pred'] in (0, 1) for p in predictions)

@torch.no_grad()
def predict_cjpe(model):
    model.eval()
    predictions = []
    for batch in tqdm(make_loader('test'), desc='CJPE test predictions'):
        labels = forward_batch(model, batch).argmax(-1).cpu().tolist()
        predictions.extend(dict(id=doc_id, pred=int(label)) for doc_id, label in zip(batch['doc_ids'], labels))
    validate_predictions(predictions, data['test'])
    return predictions

cjpe_model, cjpe_history = train_cjpe()
# Load test only after training. Its IDs are unique in the supplied data.
data['test'] = normalize_documents(
    download_json(f'{DATA_PREFIX}/Test/test.json'), labelled=False)
datasets['test'] = CJPECaseDataset(data['test'], 'test')
predictions = predict_cjpe(cjpe_model)
json_path = WORK_DIR / 'prediction.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
del cjpe_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Encoder: frozen
Frozen decoder prefix: 4
LoRMA decoder blocks: 2
Total/trainable parameters: 79705602 206082


(…)_len512_ov16_sample2_eval4/checkpoint.pt:   0%|          | 0.00/321M [00:00<?, ?B/s]

CJPE epoch 3:   0%|          | 0/1866 [00:00<?, ?it/s]

CJPE train loss=0.6785, dev macro-F1=0.3329, precision=0.2495, recall=0.5000, patience=2/10


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

... (11 outputs left out)

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Tokenizing cases:   0%|          | 0/1517 [00:00<?, ?it/s]

Cases/chunks: 1517 16594 max chunks/case: 114
Token lengths p50/p90/p99/max: [3960.0, 9699.201171875, 24538.751953125, 56511.0]


CJPE test predictions:   0%|          | 0/380 [00:00<?, ?it/s]

In [11]:
# Packaging can be rerun without retraining.
json_path = WORK_DIR / 'prediction.json'
with open(json_path, encoding='utf-8') as f: predictions = json.load(f)
validate_predictions(predictions, data['test'])
zip_path = WORK_DIR / SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname='prediction.json')
print('JSON:', json_path)
print('ZIP:', zip_path)
print('Cases:', len(predictions))


JSON: /kaggle/working/rr_runs/cjpe_encoder_decoder_mixed_lorma_len512_ov16_sample2_eval4/prediction.json
ZIP: /kaggle/working/rr_runs/cjpe_encoder_decoder_mixed_lorma_len512_ov16_sample2_eval4/cjpe_encoder_decoder_mixed_lorma_len512_ov16_sample2_eval4.zip
Cases: 1517
